[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-03-05-12-compute-memory-and-cost-arithmetic.ipynb)

# Advanced Discussion 3, 5 and 12 — training compute, memory, decode speed and serving cost

Calculators for training FLOPs and the compute-optimal split, weights and KV-cache memory, the memory-bandwidth ceiling on decode speed, and per-request cost. Prices in the notebook are placeholders.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Pure arithmetic; runs instantly.

In [ ]:
import math

## 1. compute-optimal training arithmetic (Chinchilla-style rules of thumb)

In [ ]:
def train_flops(N, D): return 6 * N * D                                      # ~6 FLOPs per parameter per training token (forward+backward)
def chinchilla_loss(N, D, E=1.69, A=406.4, B=410.7, a=0.34, b=0.28): return E + A / N ** a + B / D ** b
print("model        params   tokens(20/param)   train FLOPs    A100-days @ 40% util (312 TFLOP/s bf16)")
for name, N in (("tiny GPT", 10e6), ("small", 125e6), ("1B", 1e9), ("7B", 7e9), ("70B", 70e9)):
    D = 20 * N; F = train_flops(N, D); days = F / (0.4 * 312e12) / 86400
    print("%-9s %8.3g %14.3g %14.3g %14.1f" % (name, N, D, F, days))
print()
print("predicted loss L(N, D) = 1.69 + 406.4/N^0.34 + 410.7/D^0.28")
for N, D, note in ((7e9, 140e9, "7B, 20 tokens/param"), (7e9, 1.0e12, "7B, 143 tokens/param"), (7e9, 2.0e12, "7B, 286 tokens/param"), (70e9, 1.4e12, "70B, 20 tokens/param"), (13e9, 260e9, "13B, 20 tokens/param")):
    print("  %-24s L = %.3f" % (note, chinchilla_loss(N, D)))
# for a fixed compute budget C = 6ND, which split gives the lowest predicted loss?
C = train_flops(7e9, 140e9); best = None
for logN in [x / 100 for x in range(800, 1200)]:
    N = 10 ** logN; D = C / (6 * N); L = chinchilla_loss(N, D)
    if best is None or L < best[0]: best = (L, N, D)
print("for the 7B/140B-token budget C = %.2e FLOPs, lowest predicted loss %.3f at N = %.2e, D = %.2e (D/N = %.0f)" % (C, *best, best[2] / best[1]))

## 2. inference memory arithmetic

In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_=2): return 2 * layers * kv_heads * head_dim * bytes_
print()
print("model                 KV bytes/token   KV cache @ 4k ctx   @ 32k ctx")
for name, L_, kvh, hd in (("Llama-2-7B (MHA)", 32, 32, 128), ("Llama-3-8B (GQA, 8 KV)", 32, 8, 128), ("Llama-2-70B (GQA, 8 KV)", 80, 8, 128)):
    b = kv_bytes_per_token(L_, kvh, hd)
    print("%-24s %10d   %10.2f GB   %10.2f GB" % (name, b, b * 4096 / 1e9, b * 32768 / 1e9))
print()
print("weights memory for a 7B model:", {p: "%.1f GB" % (7e9 * by / 1e9) for p, by in (("fp16", 2), ("int8", 1), ("int4", 0.5))})
print("a free Colab T4 has 15 GB usable; Llama-2-7B fp16 needs %.1f GB weights + %.1f GB KV at 4k context for ONE sequence" % (14.0, kv_bytes_per_token(32, 32, 128) * 4096 / 1e9))
# decode speed upper bound: each new token must stream all weights through memory
print("decode upper bound tokens/s at batch 1 = memory bandwidth / weight bytes")
for gpu, bw in (("T4 (320 GB/s)", 320e9), ("A100 40GB (1,555 GB/s)", 1555e9), ("H100 (3,350 GB/s)", 3350e9)):
    print("  %-24s 7B fp16 %6.0f | int8 %6.0f | int4 %6.0f" % (gpu, bw / 14e9, bw / 7e9, bw / 3.5e9))
# batching: weights are read once per step for the whole batch
print("batch 16 on A100, 7B fp16: ~%.0f tokens/s total (if still memory-bound)" % (16 * 1555e9 / 14e9))

## 3. serving cost arithmetic (prices are ILLUSTRATIVE placeholders - use your provider's)

In [ ]:
p_in, p_out = 0.50, 1.50                                         # USD per 1M tokens (input, output) - made up for the worked example
def rag_query_cost(system=300, question=40, k=5, chunk=350, answer=250):
    tin = system + question + k * chunk; return tin, answer, (tin * p_in + answer * p_out) / 1e6
tin, tout, c = rag_query_cost()
print()
print("one RAG query: %d input tokens + %d output tokens = $%.5f; 100,000 queries/month = $%.0f" % (tin, tout, c, c * 100000))
for k in (3, 5, 10):
    _, _, ck = rag_query_cost(k=k); print("  retrieving k=%2d chunks: $%.5f per query ($%.0f per 100k)" % (k, ck, ck * 1e5))
_, _, c_cached = rag_query_cost(system=0)
print("if the 300-token system prompt were cached at zero cost: $%.5f per query (saves %.0f%%)" % (c_cached, 100 * (1 - c_cached / c)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.